# FoxTrack 01: Kaggle setup and ETL

This notebook converts the Kitsune CSV files to Parquet with Spark and finds where each attack starts.

**Before you run it (right panel of the Kaggle editor):**

1. **Input > Add Input:** add the dataset `ymirsky/network-attack-dataset-kitsune`.
2. **Settings > Internet:** set to on. The notebook installs PySpark and clones the code.
3. **Private repository only:** in **Add-ons > Secrets**, add a secret with the name `GITHUB_TOKEN` and attach it to this notebook.
4. An accelerator (GPU) is not necessary.

In [1]:
# Settings.
REPO = "DiscM/DATA-226-FoxTrack"
BRANCH = "main"
INPUT_ROOT = "/kaggle/input"
OUT_DIR = "/kaggle/working/kitsune_parquet"

In [2]:
# Spark needs Java. Install it only if it is not there.
import shutil, subprocess
if shutil.which("java") is None:
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless", shell=True, check=True)
subprocess.run(["java", "-version"], check=True)
!pip install -q pyspark mlflow

openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [3]:
# Get the project code.
import os, subprocess, sys
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None  # a public repository does not need a token
auth = f"{token}@" if token else ""
if not os.path.exists("/kaggle/working/repo"):
    done = subprocess.run(
        ["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
         f"https://{auth}github.com/{REPO}.git", "/kaggle/working/repo"],
        capture_output=True, text=True,
    )
    if done.returncode:
        raise RuntimeError("git clone failed. Examine REPO, BRANCH, and the GITHUB_TOKEN secret.")
sys.path.insert(0, "/kaggle/working/repo/src")
from foxtrack import etl

## 1. Find the files

Examine this table before the conversion. There must be nine captures, and each must have one feature file and one label file.

In [4]:
import pandas as pd
captures = etl.discover(INPUT_ROOT)
files = pd.DataFrame([{
    "attack": c.name,
    "features_file": c.features_path.name,
    "features_gb": round(c.features_path.stat().st_size / 1e9, 2),
    "labels_file": c.labels_path.name,
    **etl.sniff_features(c.features_path),
} for c in captures])
print("Total size of feature files (GB):", files["features_gb"].sum().round(2))
files

Total size of feature files (GB): 59.72


,attack,features_file,features_gb,labels_file,has_header,n_cols,has_index
0,arp_mitm,ARP_MitM_dataset.csv,7.21,ARP_MitM_labels.csv,False,115,False
1,active_wiretap,Active_Wiretap_dataset.csv,6.57,Active_Wiretap_labels.csv,False,115,False
2,fuzzing,Fuzzing_dataset.csv,6.47,Fuzzing_labels.csv,False,115,False
3,mirai_botnet,Mirai_dataset.csv,1.37,mirai_labels.csv,False,116,True
4,os_scan,OS_Scan_dataset.csv,4.89,OS_Scan_labels.csv,False,115,False
5,ssdp_flood,SSDP_Flood_dataset.csv,11.75,SSDP_Flood_labels.csv,False,115,False
6,ssl_renegotiation,SSL_Renegotiation_dataset.csv,6.36,SSL_Renegotiation_labels.csv,False,115,False
7,syn_dos,SYN_DoS_dataset.csv,7.98,SYN_DoS_labels.csv,False,115,False
8,video_injection,Video_Injection_dataset.csv,7.12,Video_Injection_labels.csv,False,115,False


## 2. Convert to Parquet

Each capture is one Spark task. Four captures run at the same time. The values are stored as 32-bit floats to stay below the 20 GB limit of `/kaggle/working`.

The conversion stops with an error if the number of feature rows and the number of labels are different.

In [5]:
%%time
spark = etl.build_spark(driver_memory="20g")
summary = etl.convert_all(spark, INPUT_ROOT, OUT_DIR, workers=4)
summary

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/21 23:08:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/21 23:08:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


CPU times: user 22.3 s, sys: 2.59 s, total: 24.9 s
Wall time: 11min 39s


,attack,rows,features,features_as_expected,dropped_index_column,null_f0_rows,malicious,malicious_fraction,first_attack_idx,raw_runs,episodes
0,arp_mitm,2504267,115,True,False,0,1145272,0.457328,1300000,49867,1
1,active_wiretap,2278689,115,True,False,0,923216,0.405152,1300000,47739,1
2,fuzzing,2244139,115,True,False,0,432783,0.192850,1300005,42362,1
3,mirai_botnet,764137,115,True,True,0,642516,0.840839,121621,1,1
4,os_scan,1697851,115,True,False,0,65700,0.038696,1314658,34069,8
5,ssdp_flood,4077266,115,True,False,0,1439604,0.353081,2613442,10324,3
6,ssl_renegotiation,2207571,115,True,False,0,92652,0.041970,1320011,29299,114
7,syn_dos,2771276,115,True,False,0,7038,0.002540,2682349,1113,8
8,video_injection,2472401,115,True,False,0,102499,0.041457,2131050,17141,3


## 3. Examine the result

Important columns in the summary:

- `features_as_expected`: must be `True` (115 features).
- `first_attack_idx`: the packet where the attack starts. All packets before it are benign.
- `episodes`: the number of attack periods. This number controls the train/test split design.
- `null_f0_rows`: must be 0. A different value shows rows that Spark could not read.

In [6]:
!du -sh {OUT_DIR}/features
df = spark.read.parquet(f"{OUT_DIR}/features")
print("Rows:", df.count(), "| Columns:", len(df.columns))
df.groupBy("attack").pivot("label").count().orderBy("attack").show()

6.4G	/kaggle/working/kitsune_parquet/features
Rows: 21017597 | Columns: 118
+-----------------+-------+-------+
|           attack|      0|      1|
+-----------------+-------+-------+
|   active_wiretap|1355473| 923216|
|         arp_mitm|1358995|1145272|
|          fuzzing|1811356| 432783|
|     mirai_botnet| 121621| 642516|
|          os_scan|1632151|  65700|
|       ssdp_flood|2637662|1439604|
|ssl_renegotiation|2114919|  92652|
|          syn_dos|2764238|   7038|
|  video_injection|2369902| 102499|
+-----------------+-------+-------+



## 4. Keep the output

Select **Save Version > Save & Run All (Commit)**. Kaggle then keeps `kitsune_parquet` as the notebook output.
In subsequent notebooks, use **Add Input > Notebook Output** to attach it. Then you do not need to run the conversion again.